In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive ETL and test validation of Supplier Invoice Oracle Financials
# Purpose: Validate ETL logic, schema, data quality, error handling, and business rules for fact_wf_supplier_invoice_orafin
# Author: Giang Nguyen
# Date: 2025-10-27
# Description: This script loads test data, applies business logic, validates schema and data types, checks error scenarios, and ensures output matches requirements for the supplier invoice fact table ETL in Databricks.

# Required PySpark imports for ETL and testing
from pyspark.sql import DataFrame 
from pyspark.sql import functions as F 
from pyspark.sql.types import ( 
    StructType, StructField, StringType, IntegerType, DoubleType, FloatType, ShortType, LongType, DateType, TimestampType
)
from pyspark.sql.utils import AnalysisException 

# -- Setup: Define test data file paths (assume files are in catalog volume, not dbfs)
test_data_paths = {
    "dw_ap_sla_aging_invoice_ca": "purgo_playground/dw_ap_sla_aging_invoice_ca.csv",
    "dw_ap_sla_expense_dist_cf": "purgo_playground/dw_ap_sla_expense_dist_cf.csv",
    "dw_party_d": "purgo_playground/dw_party_d.csv",
    "dw_supplier_site_d": "purgo_playground/dw_supplier_site_d.csv",
    "dw_internal_org_d_tl": "purgo_playground/dw_internal_org_d_tl.csv",
    "dw_ap_terms_d_tl": "purgo_playground/dw_ap_terms_d_tl.csv",
    "dw_natural_account_d": "purgo_playground/dw_natural_account_d.csv",
    "dw_ap_sla_payments_cf": "purgo_playground/dw_ap_sla_payments_cf.csv",
    "dw_gl_segment_d_tl": "purgo_playground/dw_gl_segment_d_tl.csv",
    "dw_gl_code_combination_d": "purgo_playground/dw_gl_code_combination_d.csv",
    "edp_lkup": "purgo_edp_lkp/edp_lkup.csv",
    "dim_wf_company": "purgo_dims/dim_wf_company.csv"
}

# -- Setup: Define schemas for each test table
schemas = {
    "dw_ap_sla_aging_invoice_ca": StructType([
        StructField("cost_center_segment", StringType(), True),
        StructField("gl_balancing_segment", StringType(), True),
        StructField("gl_segment1", StringType(), True),
        StructField("gl_code_combination_id", StringType(), True),
        StructField("invoiced_on_date", DateType(), True),
        StructField("invoice_id", StringType(), True),
        StructField("invoice_type_code", StringType(), True),
        StructField("transaction_currency_code", StringType(), True),
        StructField("ledger_currency_code", StringType(), True),
        StructField("invoice_schedule_due_date", DateType(), True),
        StructField("payables_bu_id", StringType(), True),
        StructField("natural_account_segment", StringType(), True),
        StructField("invoice_accounting_date", DateType(), True),
        StructField("invoice_number", StringType(), True),
        StructField("supplier_party_id", StringType(), True),
        StructField("supplier_site_id", StringType(), True),
        StructField("invoice_source_code", StringType(), True),
        StructField("snapshot_captured_date", TimestampType(), True)
    ]),
    "dw_ap_sla_expense_dist_cf": StructType([
        StructField("invoice_distribution_id", StringType(), True),
        StructField("natural_account_segment", StringType(), True),
        StructField("gl_balancing_segment", StringType(), True),
        StructField("cost_center_segment", StringType(), True),
        StructField("gl_segment1", StringType(), True),
        StructField("gl_code_combination_id", StringType(), True),
        StructField("invoice_id", StringType(), True),
        StructField("distribution_line_number", StringType(), True),
        StructField("invoice_line_number", StringType(), True),
        StructField("invoice_accounting_date", DateType(), True),
        StructField("transaction_amount", DoubleType(), True),
        StructField("xla_manual_override_flag", StringType(), True),
        StructField("invoice_description", StringType(), True),
        StructField("invoice_source_code", StringType(), True)
    ]),
    "dw_party_d": StructType([
        StructField("party_id", StringType(), True),
        StructField("party_name", StringType(), True),
        StructField("supplier_number", StringType(), True)
    ]),
    "dw_supplier_site_d": StructType([
        StructField("supplier_site_id", StringType(), True),
        StructField("address1", StringType(), True),
        StructField("address2", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("country", StringType(), True),
        StructField("PAYMENT_TERMS_ID", StringType(), True)
    ]),
    "dw_internal_org_d_tl": StructType([
        StructField("organization_id", StringType(), True),
        StructField("organization_name", StringType(), True)
    ]),
    "dw_ap_terms_d_tl": StructType([
        StructField("payment_terms_id", StringType(), True),
        StructField("payment_term_name", StringType(), True),
        StructField("payment_term_description", StringType(), True)
    ]),
    "dw_natural_account_d": StructType([
        StructField("natural_account_segment", StringType(), True)
    ]),
    "dw_ap_sla_payments_cf": StructType([
        StructField("invoice_id", StringType(), True),
        StructField("invoice_distribution_id", StringType(), True),
        StructField("check_date", DateType(), True),
        StructField("check_void_date", StringType(), True)
    ]),
    "dw_gl_segment_d_tl": StructType([
        StructField("gl_segment_code", StringType(), True),
        StructField("gl_segment_valueset_code", StringType(), True),
        StructField("gl_segment_description", StringType(), True)
    ]),
    "dw_gl_code_combination_d": StructType([
        StructField("code_combination_id", StringType(), True),
        StructField("concat_segments", StringType(), True)
    ]),
    "edp_lkup": StructType([
        StructField("lkup_key_01", StringType(), True),
        StructField("lkup_key_02", StringType(), True),
        StructField("lkup_key_03", StringType(), True),
        StructField("lkup_key_04", StringType(), True),
        StructField("lkup_typ_nm", StringType(), True),
        StructField("lkup_val_01", StringType(), True),
        StructField("lkup_val_03", StringType(), True)
    ]),
    "dim_wf_company": StructType([
        StructField("co_cd", StringType(), True),
        StructField("co_nm", StringType(), True)
    ])
}

# -- Utility function: Load CSV as DataFrame with schema and error handling
def load_csv_df(path: str, schema: StructType) -> DataFrame:
    """
    Loads a CSV file into a DataFrame with the specified schema.
    Args:
        path (str): Path to the CSV file.
        schema (StructType): Schema for the DataFrame.
    Returns:
        DataFrame: Loaded DataFrame, or raises AnalysisException if file is missing.
    """
    try:
        df = spark.read.csv(path, header=True, schema=schema, mode="FAILFAST")
        return df
    except Exception as e:
        raise AnalysisException(f"Input table not found: {path}. Error: {str(e)}")

# -- Load all test tables as DataFrames
dfs = {}
for tbl, path in test_data_paths.items():
    try:
        dfs[tbl] = load_csv_df(path, schemas[tbl])
    except AnalysisException as e:
        print(str(e))
        # For test: assert error is raised for missing table
        assert "Input table not found" in str(e)

# -- Validate schema: Ensure all required columns exist and types match
def validate_schema(df: DataFrame, schema: StructType, table_name: str):
    """
    Validates that DataFrame columns and types match the expected schema.
    Args:
        df (DataFrame): DataFrame to validate.
        schema (StructType): Expected schema.
        table_name (str): Table name for error reporting.
    Returns:
        None. Raises AssertionError if mismatch.
    """
    df_fields = {f.name: f.dataType for f in df.schema.fields}
    schema_fields = {f.name: f.dataType for f in schema.fields}
    for col, dtype in schema_fields.items():
        assert col in df_fields, f"Missing required field: {col} in {table_name}"
        assert type(df_fields[col]) == type(dtype), f"Type mismatch for {col}: expected {dtype}, got {df_fields[col]} in {table_name}"

for tbl, df in dfs.items():
    validate_schema(df, schemas[tbl], tbl)

# -- Validate allowed values for specific fields
def validate_allowed_values(df: DataFrame, field: str, allowed_values: list):
    """
    Validates that all values in a field are within allowed values.
    Args:
        df (DataFrame): DataFrame to check.
        field (str): Field name.
        allowed_values (list): List of allowed values.
    Returns:
        None. Raises AssertionError if invalid value found.
    """
    invalid = df.filter(~F.col(field).isin(allowed_values)).select(field).distinct().collect()
    for row in invalid:
        val = row[field]
        assert val is None or val in allowed_values, f"Invalid value for {field}: {val}"

validate_allowed_values(dfs["dw_ap_sla_aging_invoice_ca"], "invoice_source_code", ["ORAFIN", "MAINFRAME", "Receivables", "OTHER"])
validate_allowed_values(dfs["dw_party_d"], "supplier_number", ["10000001", "10000002", "10000003", "10000004", "99999999", "10000006", "10000007", "10000008", "10000009", "00032238", "00032239", "00080463", "90000147", None])

# -- Filter out excluded supplier numbers
excluded_supplier_numbers = ["00032238", "00032239", "00080463", "90000147"]
filtered_party_df = dfs["dw_party_d"].filter(~F.col("supplier_number").isin(excluded_supplier_numbers))
assert filtered_party_df.count() == dfs["dw_party_d"].count() - dfs["dw_party_d"].filter(F.col("supplier_number").isin(excluded_supplier_numbers)).count()

# -- Filter out excluded GL code combinations
excluded_concat_segments = [
    "4009999110011000000000000000","4009999111011104000000000000","4009999111011105000000000000",
    "4009999111011106000000000000","4009999129012946000000000000","4009999138013800000000000000",
    "4009999172017211000000000000","4009999200020004000000000000","4009999240024042000000000000",
    "4009999240024047000000000000","7009801138013800000000000000","7009801210021031000000000000",
    "7009801210021079000000000000"
]
filtered_gl_code_df = dfs["dw_gl_code_combination_d"].filter(~F.regexp_replace(F.col("concat_segments"), "\.", "") \
    .isin(excluded_concat_segments))
assert filtered_gl_code_df.count() == dfs["dw_gl_code_combination_d"].count() - dfs["dw_gl_code_combination_d"].filter(
    F.regexp_replace(F.col("concat_segments"), "\.", "").isin(excluded_concat_segments)
).count()

# -- Error handling for missing function definitions
try:
    get_basejob_url # Should be imported from ReusableFunctions
except NameError:
    raise Exception("Function not found: get_basejob_url")

try:
    read_control_table # Should be imported from ReusableFunctions
except NameError:
    raise Exception("Function not found: read_control_table")

# -- Error handling for type mismatches
def assert_type(df: DataFrame, field: str, expected_type):
    """
    Asserts that a DataFrame field is of the expected type.
    Args:
        df (DataFrame): DataFrame to check.
        field (str): Field name.
        expected_type: Expected PySpark type.
    Returns:
        None. Raises AssertionError if type mismatch.
    """
    actual_type = [f.dataType for f in df.schema.fields if f.name == field][0]
    assert type(actual_type) == type(expected_type), f"Type mismatch for {field}: expected {expected_type}, got {actual_type}"

assert_type(dfs["dw_ap_sla_expense_dist_cf"], "transaction_amount", DoubleType())
assert_type(dfs["dw_ap_sla_aging_invoice_ca"], "invoice_accounting_date", DateType())
assert_type(dfs["dw_party_d"], "supplier_party_id", StringType())

# -- Duplicate record handling: Only latest snapshot_captured_date per invoice_id
def deduplicate_invoice(df: DataFrame) -> DataFrame:
    """
    Deduplicates invoice records, keeping only the latest snapshot_captured_date per invoice_id.
    Args:
        df (DataFrame): DataFrame with invoice records.
    Returns:
        DataFrame: Deduplicated DataFrame.
    """
    window = F.window.partitionBy("invoice_id").orderBy(F.col("snapshot_captured_date").desc())
    return df.withColumn("row_num", F.row_number().over(window)).filter(F.col("row_num") == 1).drop("row_num")

deduped_invoice_df = deduplicate_invoice(dfs["dw_ap_sla_aging_invoice_ca"])
assert deduped_invoice_df.count() <= dfs["dw_ap_sla_aging_invoice_ca"].count()

# -- NULL and hardcoded field logic validation
def validate_null_or_hardcoded(df: DataFrame, field: str, value):
    """
    Validates that a field is set to NULL or a hardcoded value as required.
    Args:
        df (DataFrame): DataFrame to check.
        field (str): Field name.
        value: Expected value (None for NULL).
    Returns:
        None. Raises AssertionError if business logic missing.
    """
    vals = df.select(field).distinct().collect()
    for row in vals:
        if value is None:
            assert row[field] is None, f"Business logic missing for {field}"
        else:
            assert row[field] == value, f"Business logic missing for {field}"

# Example: document_type and txn_ref_nbr should be NULL
validate_null_or_hardcoded(deduped_invoice_df.withColumn("document_type", F.lit(None)), "document_type", None)
validate_null_or_hardcoded(deduped_invoice_df.withColumn("txn_ref_nbr", F.lit(None)), "txn_ref_nbr", None)
# Example: spend_type_cd should be 'Indirect'
validate_null_or_hardcoded(deduped_invoice_df.withColumn("spend_type_cd", F.lit("Indirect")), "spend_type_cd", "Indirect")

# -- Output DataFrame write validation
def write_output_df(df: DataFrame, path: str, table_format: str, compression: str, partition: str):
    """
    Writes the output DataFrame to the specified path with format, compression, and partitioning.
    Args:
        df (DataFrame): DataFrame to write.
        path (str): Target path.
        table_format (str): Format (delta, parquet, etc.).
        compression (str): Compression type.
        partition (str): Partition column.
    Returns:
        None. Raises AssertionError if write fails.
    """
    try:
        df.write.format(table_format).mode("overwrite").option("compression", compression).partitionBy(partition).save(path)
    except Exception as e:
        raise Exception(f"Write operation failed: {str(e)}")

# For test: Use a small sample DataFrame and validate write
sample_df = deduped_invoice_df.limit(1).withColumn("fscl_yr_nbr", F.lit("2025"))
write_output_df(sample_df, "/mnt/purgo/target_test", "delta", "snappy", "fscl_yr_nbr")

# -- Output DataFrame post-processing: Table registration in Unity Catalog
def register_table_in_unity(df: DataFrame, table_name: str, unity_catalog: str, partition: str):
    """
    Registers the DataFrame as a table in Unity Catalog with correct schema and partitioning.
    Args:
        df (DataFrame): DataFrame to register.
        table_name (str): Table name.
        unity_catalog (str): Unity Catalog name.
        partition (str): Partition column.
    Returns:
        None. Raises AssertionError if registration fails.
    """
    full_table_name = f"{unity_catalog}.{table_name}"
    try:
        df.write.format("delta").mode("overwrite").partitionBy(partition).saveAsTable(full_table_name)
    except Exception as e:
        raise Exception(f"Table registration failed: {str(e)}")
    # Validate table exists and schema matches
    loaded_df = spark.table(full_table_name)
    assert set([f.name for f in loaded_df.schema.fields]) == set([f.name for f in df.schema.fields]), "Schema mismatch after table registration"

register_table_in_unity(sample_df, "fact_wf_supplier_invoice_orafin_test", "purgo_databricks", "fscl_yr_nbr")

# -- Performance test: Validate ETL can process batch and streaming scenarios
def performance_test_batch(df: DataFrame):
    """
    Measures batch ETL performance for a DataFrame.
    Args:
        df (DataFrame): DataFrame to process.
    Returns:
        float: Time taken in seconds.
    """
    import time 
    start = time.time()
    result = df.groupBy("invoice_id").count().collect()
    end = time.time()
    assert len(result) > 0, "Batch ETL produced no results"
    return end - start

batch_time = performance_test_batch(deduped_invoice_df)
print(f"Batch ETL time: {batch_time:.2f} seconds")

def performance_test_streaming(df: DataFrame):
    """
    Measures streaming ETL performance for a DataFrame.
    Args:
        df (DataFrame): DataFrame to process.
    Returns:
        None. Asserts streaming query runs.
    """
    try:
        stream_df = df.writeStream.format("memory").queryName("test_stream").outputMode("append").start()
        stream_df.processAllAvailable()
        result = spark.sql("SELECT * FROM test_stream").count()
        assert result > 0, "Streaming ETL produced no results"
        stream_df.stop()
    except Exception as e:
        print(f"Streaming test error: {str(e)}")

performance_test_streaming(deduped_invoice_df.limit(10))

# -- Data quality validation tests
def validate_no_nulls(df: DataFrame, required_fields: list):
    """
    Validates that required fields are not NULL.
    Args:
        df (DataFrame): DataFrame to check.
        required_fields (list): List of required field names.
    Returns:
        None. Raises AssertionError if NULLs found.
    """
    for field in required_fields:
        null_count = df.filter(F.col(field).isNull()).count()
        assert null_count == 0, f"Missing required field: {field}"

validate_no_nulls(sample_df, ["invoice_id", "invoice_accounting_date", "supplier_party_id", "supplier_site_id", "invoice_number"])

# -- Delta Lake operations: MERGE, UPDATE, DELETE validation
from delta.tables import DeltaTable 

def delta_merge_update_delete_test(df: DataFrame, table_name: str, unity_catalog: str):
    """
    Tests Delta Lake MERGE, UPDATE, DELETE operations.
    Args:
        df (DataFrame): DataFrame to merge/update/delete.
        table_name (str): Table name.
        unity_catalog (str): Unity Catalog name.
    Returns:
        None. Asserts operations succeed.
    """
    full_table_name = f"{unity_catalog}.{table_name}"
    # Register table
    df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)
    delta_tbl = DeltaTable.forName(spark, full_table_name)
    # MERGE: Upsert a record
    merge_df = df.limit(1).withColumn("unit_prc", F.lit(9999.99))
    delta_tbl.alias("tgt").merge(
        merge_df.alias("src"),
        "tgt.invoice_id = src.invoice_id"
    ).whenMatchedUpdate(set={"unit_prc": F.col("src.unit_prc")}) \
     .whenNotMatchedInsertAll().execute()
    # UPDATE: Set unit_prc to 8888.88 for one record
    delta_tbl.update(condition="invoice_id = '{}'".format(merge_df.collect()[0]["invoice_id"]),
                     set={"unit_prc": F.lit(8888.88)})
    # DELETE: Remove the record
    delta_tbl.delete(condition="invoice_id = '{}'".format(merge_df.collect()[0]["invoice_id"]))
    # Validate record deleted
    assert delta_tbl.toDF().filter(F.col("invoice_id") == merge_df.collect()[0]["invoice_id"]).count() == 0, "Delta DELETE failed"

delta_merge_update_delete_test(sample_df, "fact_wf_supplier_invoice_orafin_delta_test", "purgo_databricks")

# -- Window function and analytics feature test
def window_analytics_test(df: DataFrame):
    """
    Tests window functions for analytics (row_number, sum, etc.).
    Args:
        df (DataFrame): DataFrame to test.
    Returns:
        None. Asserts window function results.
    """
    from pyspark.sql.window import Window 
    w = Window.partitionBy("invoice_id").orderBy(F.col("snapshot_captured_date").desc())
    df2 = df.withColumn("row_num", F.row_number().over(w))
    assert df2.filter(F.col("row_num") == 1).count() > 0, "Window function row_number failed"
    df3 = df2.groupBy("invoice_id").agg(F.sum("row_num").alias("row_num_sum"))
    assert df3.count() > 0, "Window function sum failed"

window_analytics_test(deduped_invoice_df)

# -- Cleanup operations: Drop test tables
def cleanup_test_tables(table_names: list, unity_catalog: str):
    """
    Drops test tables from Unity Catalog.
    Args:
        table_names (list): List of table names.
        unity_catalog (str): Unity Catalog name.
    Returns:
        None.
    """
    for tbl in table_names:
        full_table_name = f"{unity_catalog}.{tbl}"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
        except Exception as e:
            print(f"Cleanup error for {full_table_name}: {str(e)}")

cleanup_test_tables([
    "fact_wf_supplier_invoice_orafin_test",
    "fact_wf_supplier_invoice_orafin_delta_test"
], "purgo_databricks")

# END OF SCRIPT